# An Open-source Supported Guide to High-speed Camera Based Structural Dynamics Identification

**D. Gorjup, K. Zaletelj, J. Slavič**  
University of Ljubljana, Faculty of Mechanical Engineering

*Mechanical Systems and Signal Processing*, TODO (TODO). DOI: [TODO](https://doi.org/TODO)

---

This notebook provides a complete, end-to-end worked example of the methods described in the paper above.
Each section maps directly to the corresponding paper section.

**How to cite:**
```bibtex
@article{gorjup2026tutorial,
  author  = {Gorjup, Domen and Zaletelj, Klemen and Slavi\v{c}, Janko},
  title   = {An Open-source Supported Guide to High-speed Camera Based Structural Dynamics Identification},
  journal = {Mechanical Systems and Signal Processing},
  year    = {TODO},
  volume  = {TODO},
  pages   = {TODO},
  doi     = {TODO},
}
```

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

---
## 2  Practical Considerations for Image-Based Vibration Measurement

This section covers the experimental setup steps that precede any displacement identification:
camera selection and configuration, geometric calibration, surface preparation, and an
overview of the main sources of measurement error.

### 2.1  Camera Selection and Setup

Key parameters: sensor resolution, frame rate, dynamic range, shutter type (global vs. rolling).
The cell below illustrates the distortion caused by a rolling shutter on a sinusoidally vibrating object.

In [ ]:
# ex_2_3_shutter_comparison
# TODO: simulate / load rolling-shutter vs. global-shutter frames and visualise the skew artefact

### 2.2  Camera Calibration

The pinhole model relates a 3-D world point **X** to its image projection **x** via
the intrinsic matrix **K** and the extrinsic pose (R, t).  
Calibration is performed with a planar checkerboard target; quality is assessed by the
reprojection error.

In [ ]:
# ex_2_1_camera_calibration
# TODO: load checkerboard image sequence, detect corners, estimate K / distortion coefficients,
#       report mean reprojection error

### 2.3  Lighting and Surface Preparation

Adequate, uniform illumination and a high-contrast speckle pattern are prerequisites for
reliable sub-pixel displacement estimation.  The histogram below illustrates a well-exposed
speckle pattern.

In [ ]:
# TODO: load a sample speckle image and plot its intensity histogram

### 2.4  Sources of Measurement Error

Primary error sources: image noise, blur (motion and defocus), aliasing, lens distortion,
and illumination variation.

In [ ]:
# TODO: demonstrate effect of one or more error sources (e.g. SNR vs. displacement bias)

---
## 3  Identifying Displacements in Digital Images

Two complementary paradigms are presented: **optical flow** (Eulerian, dense or sparse)
and **Digital Image Correlation** (Lagrangian, subset-based).  Both are implemented in
[pyIDI](https://github.com/ladisk/pyidi).

### 3.1  Optical Flow Displacement Identification

The brightness constancy assumption yields the optical flow constraint equation:

$$\frac{\partial I}{\partial x} u + \frac{\partial I}{\partial y} v + \frac{\partial I}{\partial t} = 0$$

The Lucas–Kanade method resolves the aperture problem by assuming uniform flow within a
small spatial window and solving the resulting over-determined system in a least-squares sense.

In [ ]:
import pyidi

# TODO: load video with pyidi.VideoReader, initialise LucasKanade method,
#       set_points(), run get_displacements(), plot displacement time series

### 3.2  Digital Image Correlation (DIC)

DIC tracks a subset of pixels by maximising a normalised cross-correlation (ZNCC) or
minimising a sum-of-squared-differences (SSD / ZNSSD) criterion over deformation parameters
described by a shape function.

In [ ]:
# TODO: demonstrate subset-based DIC with pyIDI; compare similarity measures (SSD, ZNCC)

### 3.3  Similarity Measures

The choice of similarity measure affects robustness to illumination changes and noise.
ZNSSD / ZNCC are invariant to linear intensity shifts and scaling.

In [ ]:
# ex_4_2_subset_size
# TODO: sweep subset size and plot the trade-off between spatial resolution and noise

### 3.4  Image-based Displacement Identification for High-Frequency Vibration

At high vibration frequencies relative to the frame rate, amplitude and phase estimates
can be recovered from sub-Nyquist aliased signals if the excitation is periodic and the
aliasing pattern is known.

In [ ]:
# TODO: demonstrate high-frequency identification from aliased camera data

---
## 4  Measuring 3D Displacements from Digital Images

A single camera provides 2-D projections only.  3-D displacements are recovered either via
**stereo triangulation** (geometric, using calibrated camera pairs) or **frequency-domain
triangulation** (exploiting the out-of-plane to in-plane displacement ratio at resonance).

### 4.1  Multi-view Imaging and Triangulation

The Direct Linear Transform (DLT) assembles, for each camera view, two linear equations
in the unknown 3-D point **X**.  With ≥ 2 views the over-determined system is solved by SVD.

In [ ]:
# TODO: given two sets of 2-D displacement time series (camera 1 and 2) and calibration
#       matrices P1, P2, assemble DLT system and recover 3-D displacements via SVD

### 4.2  Frequency-Domain Triangulation

At each resonance frequency the 3-D mode shape vector can be recovered from the complex
spectra of the 2-D projected displacements, using the known projection matrices.

In [ ]:
# TODO: compute displacement spectra from multiple camera views, apply frequency-domain
#       triangulation at identified resonance frequencies, visualise 3-D deflection shapes

### 4.3  When Is 3D Measurement Necessary?

For planar structures or in-plane dominant modes a single camera is sufficient.
This cell provides a simple criterion based on the expected out-of-plane to in-plane ratio.

In [ ]:
# TODO: quantitative example or figure illustrating when 3-D reconstruction adds value

---
## 5  Experimental Modal Analysis Using Image-based Measurements

Camera-derived displacement time series are used to estimate FRFs, extract modal parameters
(natural frequencies, damping, mode shapes), and validate them against reference data.

### 5.1  Frequency Response Functions

H₁ and H₂ FRF estimators are computed from cross- and auto-spectral density matrices
using Welch's averaged periodogram method.  Coherence provides a quality indicator.

In [ ]:
# ex_6_1_frf_estimation
# TODO: load force and displacement signals, compute H1 / H2 estimators and coherence,
#       plot FRF magnitude and phase

### 5.2  The Hybrid Modal Identification Method

Natural frequencies and damping ratios are identified from high-quality accelerometer FRFs
using LSCF (poly-reference Least-Squares Complex Frequency).  Mode shapes are then extracted
at those poles from the full camera FRF matrix using LSFD.

In [ ]:
import sdypy as sd

# TODO: run LSCF on accelerometer FRFs to obtain poles (natural frequencies + damping),
#       then LSFD on camera FRFs to extract full-field mode shapes

### 5.3  Validating Camera-based Mode Shapes Using the Modal Assurance Criterion

The Modal Assurance Criterion (MAC) quantifies the correlation between two mode shape vectors:

$$\text{MAC}(\boldsymbol{\phi}_r, \boldsymbol{\psi}_r) =
\frac{|\boldsymbol{\phi}_r^\mathrm{H} \boldsymbol{\psi}_r|^2}
{(\boldsymbol{\phi}_r^\mathrm{H} \boldsymbol{\phi}_r)(\boldsymbol{\psi}_r^\mathrm{H} \boldsymbol{\psi}_r)}$$

A value close to 1 indicates good agreement; off-diagonal values should be near zero.

In [ ]:
# TODO: compute MAC matrix between camera-based and reference (accelerometer / FE) mode shapes,
#       plot as a colour matrix